In [2]:
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor07.R")
download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R")
source("config.R")
cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"),
                   stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs   <- setNames(cp$texto, cp$id)
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))
ix     <- montar(docs, cfg)
estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 20:05 · R version 4.6.1 (2026-06-24) · motor07 v1 2026-10-15 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 12 | de, e, a, o, em, da, ... |
| `cfg$grupo` | character | 1 | Predicarga |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 1 |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `cp` | data.frame | 64 × 5 | colunas: id, titulo, texto, fonte, url |
| `docs` | character, nomeado | 64 | d1 São Vicente, oficialmente E..., d2 São Vicente marca o início ..., d3 Com um PIB de mais de 6 bil..., d4 São Vicente preserva divers..., d5 A Ilha de Gohayó foi descob..., d6 Nas primeiras décadas do sé

In [3]:
arqs <- c("qrels_Andre_2026-10-25.csv")
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)
qrels <- ler_qrels(arqs, juiz = cfg$juizes)
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"),
                  stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]
names(qrels)[R_q == 0]

character(0)

In [4]:
names(qrels)
usar
R_q

[1] "q01" "q02" "q03" "q04" "q05"

[1] "q01" "q02" "q03" "q04" "q05"

q01 q02 q03 q04 q05 
  1   5   2   6   6

In [5]:
dim(ix$w); min(dim(ix$w))
m20 <- lsa(ix, 20)
s <- svd(ix$w)
round(sum(m20$S^2) / sum(s$d^2), 3)

[1] 1831   64

[1] 64

ERROR: Error in lsa(ix, 20): could not find function "lsa"


In [6]:
# Módulo 3 — LSA: SVD truncada da matriz TF-IDF
lsa <- function(ix, k) {                 # k: quantas dimensões guardar
  s <- svd(ix$w)                         # W = U Sigma V^t
  U <- s$u[, 1:k, drop = FALSE]          # termos x k
  rownames(U) <- ix$vocab
  D <- s$v[, 1:k, drop = FALSE] %*% diag(s$d[1:k], nrow = k)   # documentos x k
  rownames(D) <- colnames(ix$w)
  list(U = U, S = s$d[1:k], D = D)       # o modelo
}

# Módulo 4 — dobrar a consulta e ranquear por cosseno no espaço LSA
ranking_lsa <- function(consulta, ix, cfg, modelo) {
  q0 <- vetor_consulta(preparar(consulta, cfg), ix$vocab, ix$idf_tfidf)  # texto -> pesos
  qk <- t(modelo$U) %*% q0               # dobra a consulta para k coordenadas
  s <- apply(modelo$D, 1, function(d) cosseno(qk, d))   # um cosseno por documento
  sort(s, decreasing = TRUE)             # do maior para o menor
}

# Módulo 6 — fusão RRF de rankings
rrf <- function(..., c = 60) {           # c amortece a diferença entre posições
  rks <- list(...)                       # os rankings que vierem
  ids <- unique(unlist(lapply(rks, names)))
  s <- sapply(ids, function(d)
    sum(sapply(rks, function(r) {
      p <- which(names(r) == d)
      if (length(p) == 1) 1 / (c + p) else 0
    })))
  sort(s, decreasing = TRUE)
}

In [7]:
m20 <- lsa(ix, 20)                                # LSA de trabalho, k = 20
s <- svd(ix$w)                                    # SVD inteira, só para a energia
round(sum(m20$S^2) / sum(s$d^2), 3)               # fração da energia guardada

[1] 0.616

In [8]:
q  <- usar[1]
rl <- ranking_lsa(consultas[[q]], ix, cfg, m20)
rb <- ranking_bm25(consultas[[q]], ix, cfg)
round(head(rl, 5), 3)
rbind(bm25 = names(rb)[1:5], lsa = names(rl)[1:5])
so_lsa <- names(rl)[1:5][!names(rl)[1:5] %in% names(rb)[1:5]]
docs[so_lsa]

d10   d12    d6   d39    d7 
0.789 0.783 0.758 0.738 0.711

bm25,d6,d2,d4,d1,d3
lsa,d10,d12,d6,d39,d7


d10 
                                                                                                                                                                                                                                                                                                                                     "No início da colonização portuguesa, o Padre José de Anchieta contribuiu na catequização dos índios e na harmonização do povoado através do colégio vicentino. Ele ministrou suas aulas de catecismo junto à denominada \"Bica da Fonte do Povoado\", que, atualmente, é fonte histórica denominada afetivamente como \"Biquinha\"." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       d12 
                                                                                                                                                                                     "No final do ano de 1541, ocorreu o pior desastre natural em São Vicente, quando uma onda gigante, um possível maremoto, danificou seriamente a vila, destruindo essa povoação, que abrigava um total de 150 habitantes. O porto, que funcionava como fonte da economia, mudou-se para onde está hoje. A Igreja Matriz, a Casa do Conselho, a Cadeia, os estaleiros, o pelourinho e inúmeras casas foram destruídos. A Vila teve que ser reconstruída um pouco mais distante do mar." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       d39 
                                                                                                                                                                                                                                                                                                                                                                                                            "Uma onda gigante, um possível maremoto, no final do ano de 1541, também danificou seriamente essa vila. A catástrofe destruiu o lugar que abrigava um total de 150 habitantes. O porto, que funcionava como fonte da economia, mudou-se para onde está hoje." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        d7 
"Entre 1530 e 1532, a expedição liderada pelo fidalgo português Martim Afonso de So

In [9]:
sapply(so_lsa, function(d)
  length(intersect(preparar(consultas[[q]], cfg), ix$tokens[[d]])))
qrels[[q]][so_lsa]

d10 d12 d39  d7 
  0   0   0   0

<NA> <NA> <NA> <NA> 
  NA   NA   NA   NA

In [10]:
ranking_modelo <- function(txt, modelo, ix, cfg, m, c = 60) {
  rb <- ranking_bm25(txt, ix, cfg)
  if (modelo == "bm25") return(rb)
  rl <- ranking_lsa(txt, ix, cfg, m)
  if (modelo == "lsa") rl else rrf(rb, rl, c = c)
}

ap_modelo <- function(q, modelo, m, consultas, qrels, ix, cfg)
  metricas(ranking_modelo(consultas[[q]], modelo, ix, cfg, m),
           qrels[[q]], cfg$limiar)[["AP"]]

In [11]:
ks <- c(10, 20, 50)
ks <- ks[ks <= min(dim(ix$w))]
ms <- setNames(lapply(ks, function(k) lsa(ix, k)), paste0("k", ks))
names(ms)

[1] "k10" "k20" "k50"

In [12]:
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])

tab_dev <- sapply(dev, function(q) c(
  bm25 = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),
  lsa  = sapply(ms, function(m) ap_modelo(q, "lsa", m, consultas, qrels, ix, cfg)),
  hib  = sapply(ms, function(m) ap_modelo(q, "hibrido", m, consultas, qrels, ix, cfg))))

round(tab_dev, 3)
round(rowMeans(tab_dev), 3)

,q01,q02,q03
bm25,0.023,0.097,0.079
lsa.k10,0.019,0.094,0.048
lsa.k20,0.017,0.080,0.079
lsa.k50,0.020,0.078,0.137
hib.k10,0.019,0.157,0.063
hib.k20,0.018,0.082,0.081
hib.k50,0.019,0.081,0.102


bm25 lsa.k10 lsa.k20 lsa.k50 hib.k10 hib.k20 hib.k50 
  0.066   0.054   0.058   0.078   0.080   0.060   0.067

In [13]:
escolha <- "bm25"
k_esc <- 20
m_esc <- lsa(ix, k_esc)

tab_teste <- sapply(teste, function(q) c(
  bm25      = ap_modelo(q, "bm25", NULL, consultas, qrels, ix, cfg),
  escolhido = ap_modelo(q, escolha, m_esc, consultas, qrels, ix, cfg)))

round(tab_teste, 3)
round(rowMeans(tab_teste), 3)

,q04,q05
bm25,0.112,0.071
escolhido,0.112,0.071


bm25 escolhido 
    0.092     0.092

In [14]:
  # --- Aula 07: LSA e busca hibrida --------------------------------------
  modelo = "bm25",                        # "bm25", "lsa" ou "hibrido" (RRF de bm25 e lsa)
  lsa_k  = 20,                            # dimensoes do LSA
  rrf_c  = 60,                            # constante da fusao RRF

ERROR: Error in parse(text = input): <text>:2:18: unexpected ','
1:   # --- Aula 07: LSA e busca hibrida --------------------------------------
2:   modelo = "bm25",
                    ^


In [15]:
source("config.R")
cfg$modelo
cfg$lsa_k
cfg$rrf_c

[1] "bm25"

[1] 20

[1] 60

In [16]:
ranking_motor <- function(txt, ix, cfg)
  ranking_modelo(txt, cfg$modelo, ix, cfg, lsa(ix, cfg$lsa_k), c = cfg$rrf_c)

round(head(ranking_motor(consultas[[usar[1]]], ix, cfg), 5), 4)

d6     d2     d4     d1     d3 
3.5114 2.2963 2.0597 0.0000 0.0000

In [17]:
cfg$modelo
cfg$lsa_k
cfg$rrf_c

[1] "bm25"

[1] 20

[1] 60

In [18]:
anexar_estado("00_FICHA_PROJETO.md")
anexar_estado("aula07_parteD_consolidado.md")

Estado do R anexado a 00_FICHA_PROJETO.md (64 linhas de objetos).

Estado do R anexado a aula07_parteD_consolidado.md (64 linhas de objetos).

